<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B02%5D%20-%20Analisis_Cluster/%5B01%5D%20-%20Notebooks/E4_K_means_con_IRIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K-Means con IRIS Dataset


En este ejercicio práctico de Machine Learning no supervisado, trabajaremos con el famoso conjunto de datos Iris para aplicar el algoritmo de clustering K-Means.

El dataset contiene 150 observaciones de flores de tres especies distintas (Setosa, Versicolor y Virginica), caracterizadas por cuatro variables numéricas: longitud y anchura de sépalo y pétalo.

A lo largo del ejercicio, abordaremos todo el flujo completo: desde la carga y exploración de los datos, su preprocesamiento, la determinación óptima del número de clústeres, la creación del modelo K-Means y su interpretación.

Además, realizaremos una evaluación cruzada con las etiquetas reales para entender la capacidad del modelo de separar clases, y finalizaremos clasificando nuevas observaciones basándonos en los clústeres aprendidos.

Este enfoque permite entender cómo un modelo no supervisado puede usarse como base para tareas de clasificación, incluso sin tener acceso directo a las etiquetas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
# Carga del dataset Iris
iris = datasets.load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['true_class'] = iris.target

## 2. Exploración y visualización

In [ ]:
# Matriz de scatter plot
pd.plotting.scatter_matrix(df.iloc[:, :4], figsize=(10,10), diagonal='kde')
plt.show()

## 3. Preprocesado: escalado de variables

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df.iloc[:, :4])
# Escalar evita que variables con rangos mayores dominen la distancia euclídea.


## 4. Determinar el número óptimo de clústeres (método “codo” y Silhouette)

In [ ]:
inertia, sil = [], []
K = range(2, 7)
for k in K:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42)
    km.fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_))

plt.plot(K, inertia, '-o'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Método del codo')
plt.show()

plt.plot(K, sil, '-o'); plt.xlabel('k'); plt.ylabel('Silhouette'); plt.title('Coef. Silhouette')
plt.show()


## 5. Entrenar el modelo final

In [ ]:
k_opt = 5
kmeans = Pipeline([
    ('scale', StandardScaler()),
    ('km', KMeans(n_clusters=k_opt, init='k-means++', random_state=42))
])
y_pred = kmeans.fit_predict(df.iloc[:, :4])
df['cluster'] = y_pred

## 6. Evaluación y patrones

In [ ]:
table = pd.crosstab(df['cluster'], df['true_class'], rownames=['cluster'], colnames=['true_class'])
print(table)


# reasignamos clusters al label más común dentro de cada cluster
mapping = {i: df[df.cluster==i].true_class.mode()[0] for i in df.cluster.unique()}
df['cluster_aligned'] = df.cluster.map(mapping)
print("\n\n")
print("Accuracy cluster vs true: ", round(accuracy_score(df.true_class, df.cluster_aligned), 1))
print("\n\n")
print(classification_report(df.true_class, df.cluster_aligned))

## 7. Visualización de clústeres

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X[:,2], X[:,3], c=y_pred, cmap='viridis', alpha=0.6)
plt.xlabel('Petal length (scaled)'); plt.ylabel('Petal width (scaled)')
plt.title('Clústeres K‑means')
plt.show()

## 8. Clasificar nuevas observaciones

In [ ]:
# Supongamos nueva muestra:
new = np.array([[5.1, 3.5, 1.4, 0.2]])
cluster_new = kmeans.predict(new)[0]
class_new = mapping[cluster_new]
print(f"Nueva muestra asignada al clúster {cluster_new}, clase aproximada: {iris.target_names[class_new]}")

## 9. Bonus: automatizar pipeline completo

In [ ]:
# Dividir dataset para simular entrenamiento y clasificación:
X_train, X_test, y_true_train, y_true_test = train_test_split(df.iloc[:, :4], df.true_class, test_size=0.3, random_state=42)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('km', KMeans(n_clusters=3, init='k-means++', random_state=42))
])
y_train_cluster = pipe.fit_predict(X_train)
# Mapeo igual que antes
mapping2 = {i: y_true_train[y_train_cluster==i].mode()[0] for i in np.unique(y_train_cluster)}
y_pred_test = pipe.predict(X_test)
y_pred_labels = pd.Series(y_pred_test).map(mapping2)

print(classification_report(y_true_test, y_pred_labels, target_names=iris.target_names))